# Media M&A Valuation: Paramount / Warner Bros. Discovery Style

## Educational portfolio case — model figures are fictional

This notebook accompanies the corrected Excel workbook. The historical figures, assumptions, projections, peers, offer price, debt financing, synergies and EPS outputs are invented placeholders. They do **not** represent actual reported figures or the economics of any real transaction.

The workbook includes a separate `Public Deal Facts` sheet for publicly reported context. Never conflate that context with this hypothetical model.

**Currency:** USD billions unless otherwise stated. Share counts are in billions; EPS and share price are in USD per share.

## 1. How to run

Install the basic dependencies from the project folder:

```bash
pip install pandas numpy matplotlib
```

Run the Python validation script as well:

```bash
python media_ma_validation.py
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from media_ma_validation import Case, calculate_wacc, forecast, dcf_value, dcf_sensitivity, comps_math, transaction_math, eps_math

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
c = Case()
cost_equity, wacc = calculate_wacc(c)
print(f"Cost of equity: {cost_equity:.2%}")
print(f"WACC: {wacc:.2%}")
print("These are outputs of fictional assumptions, not market-derived deal metrics.")

## 2. Forecast: revenue, EBITDA and free cash flow

Revenue starts from the fictional 2025 combined revenue of $66bn and grows at 2.0%–3.0%. EBITDA margins are set to 12.5% in 2026, gradually increasing to 14.5% in 2030. This is a modest assumed recovery path; it is not management guidance.

In [ ]:
fc = forecast(c)
fc[["Year", "Revenue ($bn)", "Revenue growth", "EBITDA ($bn)", "EBITDA margin", "EBIT ($bn)", "NOPAT ($bn)", "Capex ($bn)", "Change in NWC ($bn)", "Unlevered FCFF ($bn)"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(fc["Year"], fc["Revenue ($bn)"], marker="o", label="Revenue")
ax.plot(fc["Year"], fc["EBITDA ($bn)"], marker="o", label="EBITDA")
ax.set_title("Illustrative Combined Company Forecast")
ax.set_ylabel("USD billions")
ax.set_xlabel("Fiscal year")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 3. DCF and terminal value

The model discounts five years of unlevered free cash flow using WACC. It then normalizes terminal-year D&A to capex, a steady-state simplification intended to avoid carrying a permanent D&A-over-capex gap into perpetuity. This is a modelling choice, not an accounting forecast.

In [ ]:
normalized_terminal_fcff = dcf_sensitivity(
    fc["Unlevered FCFF ($bn)"].values,
    revenue_2030=fc["Revenue ($bn)"].iloc[-1],
    revenue_2029=fc["Revenue ($bn)"].iloc[-2],
    ebitda_margin_2030=c.ebitda_margins[-1],
    tax=c.tax,
    capex_pct=c.capex_pct_revenue,
    nwc_pct=c.nwc_pct_revenue_change,
)
dcf = dcf_value(fc["Unlevered FCFF ($bn)"].values, wacc, c.perpetual_growth,
                terminal_fcff_normalized=normalized_terminal_fcff)
pd.Series({**dcf, "Perpetual growth": c.perpetual_growth, "Cost of equity": cost_equity, "WACC": wacc})

## 4. Trading comparables

The peer multiple set is fictional and is provided to demonstrate mechanics. EV/EBITDA can be useful for comparing companies with different capital structures, but peer selection must account for business mix, margin, growth, content obligations and accounting definitions. Target equity value is a thin residual when net debt is high.

In [ ]:
comps = comps_math(c)
pd.Series(comps)

## 5. Offer price and sources & uses

The illustrative offer includes a 20% premium to the comps-derived target equity proxy. The model funds uses with new acquisition debt, stock consideration and buyer cash / other equity funding. Debt refinanced plus debt retained reconciles to the fictional target net debt.

In [ ]:
transaction = transaction_math(c, comps["Target implied equity value ($bn)"])
pd.Series(transaction)
assert abs(transaction["Sources less uses ($bn)"]) < 1e-8
assert abs(transaction["Debt reconciliation check ($bn)"]) < 1e-8

## 6. EPS accretion / dilution

The buyer and target are loss-making on the fictional 2025 inputs, so percentage EPS accretion/dilution is marked not meaningful. The dollar-per-share change is shown instead. Incremental interest includes interest on new acquisition debt, foregone interest on buyer cash, and interest saved on refinanced target debt. One-time integration costs are also shown separately.

In [ ]:
eps = eps_math(c, transaction)
pd.Series(eps)

## 7. Sensitivity analysis

Explore how DCF enterprise value changes with WACC and perpetual growth. Each cell is a model scenario, not a forecast or real transaction value.

In [ ]:
wacc_grid = [0.075, 0.080, 0.085, 0.090, 0.095]
growth_grid = [0.010, 0.020, 0.025, 0.030, 0.035]
rows = []
for w in wacc_grid:
    row = []
    for g in growth_grid:
        val = dcf_value(fc["Unlevered FCFF ($bn)"].values, w, g,
                        terminal_fcff_normalized=normalized_terminal_fcff)["DCF enterprise value ($bn)"]
        row.append(val)
    rows.append(row)
sensitivity = pd.DataFrame(rows, index=[f"WACC {w:.1%}" for w in wacc_grid],
                           columns=[f"Growth {g:.1%}" for g in growth_grid])
sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(sensitivity.values, aspect="auto")
ax.set_xticks(range(len(sensitivity.columns)), sensitivity.columns)
ax.set_yticks(range(len(sensitivity.index)), sensitivity.index)
ax.set_title("DCF Enterprise Value Sensitivity (USD bn)")
for i in range(sensitivity.shape[0]):
    for j in range(sensitivity.shape[1]):
        ax.text(j, i, f"{sensitivity.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Enterprise value ($bn)")
plt.tight_layout()
plt.show()

## 8. Model interpretation and limitations

- A higher WACC lowers DCF value; a higher perpetual growth rate raises terminal value. Small changes to terminal assumptions can materially affect value.
- The target equity value is highly sensitive to EV/EBITDA because it is calculated as enterprise value less net debt.
- EPS accretion is not the same as value creation. A deal can be EPS-accretive and still destroy value if the buyer overpays or cannot realize synergies.
- This model omits a complete debt amortization schedule, purchase accounting, detailed content obligations, minimum cash, tax attributes, detailed integration timing and actual market data.
- To turn this into real-company research, source all historical data, market prices, share counts, debt schedules and peer multiples as of a clearly specified date. Cite each input and maintain a source log.

## 9. Official public context

The workbook's `Public Deal Facts` sheet includes official source links for the public transaction context. Any targets cited there are forward-looking management targets, not guaranteed results. They do not make this fictional model's assumptions real.